In [ ]:
import numpy as np
from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
from bokeh.models import CustomJS, Slider, ColumnDataSource
from bokeh.plotting import figure

In [ ]:
output_notebook()

In [ ]:
E = np.linspace(0.1, 80, 800)
kT_init = 4.0
b_init = 12.0

f_MB = np.exp(-E / kT_init)
f_tunnel = np.exp(-b_init / np.sqrt(E))
f_Gamow = f_MB * f_tunnel


f_MB_norm = f_MB / np.max(f_MB)
f_tunnel_norm = f_tunnel / np.max(f_tunnel)
f_Gamow_norm = f_Gamow / np.max(f_Gamow)

source = ColumnDataSource(data=dict(
    E=E,
    mb=f_MB_norm,
    tunnel=f_tunnel_norm,
    gamow=f_Gamow_norm
))


plot = figure(
    title="Gamow Peak", 
    x_axis_label="Energy (E)", 
    y_axis_label="Normalized Intensity",
    x_range=(0, 80), 
    y_range=(0, 1.1),
    width=750, 
    height=450
)

plot.line('E', 'mb', source=source, color="blue", line_width=2, line_dash="dashed", legend_label="Maxwell-Boltzmann")
plot.line('E', 'tunnel', source=source, color="red", line_width=2, line_dash="dotdash", legend_label="Penetration Prob.")
plot.line('E', 'gamow', source=source, color="purple", line_width=3.5, legend_label="Gamow Peak")


plot.legend.location = "top_right"
plot.legend.click_policy = "hide" 


slider_kT = Slider(start=1.0, end=15.0, value=kT_init, step=0.1, title="Temperature factor (kT)")
slider_b = Slider(start=5.0, end=25.0, value=b_init, step=0.1, title="Barrier / Charge factor (b)")


callback = CustomJS(args=dict(source=source, slider_kT=slider_kT, slider_b=slider_b), code="""
    const data = source.data;
    const kT = slider_kT.value;
    const b = slider_b.value;
    const E = data['E'];
    const mb = data['mb'];
    const tunnel = data['tunnel'];
    const gamow = data['gamow'];

    let max_mb = 0;
    let max_tunnel = 0;
    let max_gamow = 0;

    for (let i = 0; i < E.length; i++) {
        mb[i] = Math.exp(-E[i] / kT);
        tunnel[i] = Math.exp(-b / Math.sqrt(E[i]));
        gamow[i] = mb[i] * tunnel[i];

        if (mb[i] > max_mb) max_mb = mb[i];
        if (tunnel[i] > max_tunnel) max_tunnel = tunnel[i];
        if (gamow[i] > max_gamow) max_gamow = gamow[i];
    }

    for (let i = 0; i < E.length; i++) {
        mb[i] = max_mb > 0 ? mb[i] / max_mb : 0;
        tunnel[i] = max_tunnel > 0 ? tunnel[i] / max_tunnel : 0;
        gamow[i] = max_gamow > 0 ? gamow[i] / max_gamow : 0;
    }

    source.change.emit();
""")

# Sliders to JS Callback
slider_kT.js_on_change('value', callback)
slider_b.js_on_change('value', callback)


layout = column(row(slider_kT, slider_b, width=750), plot)
show(layout)